In [ ]:
import torch
import os
import torch.nn as nn
import torchvision.datasets
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score
import pandas as pd

from Network import CifarNeural

In [2]:
csv_filename = "training_chart_data.csv"
fine_tune = False

if fine_tune or not os.path.exists(csv_filename):
  start_epoch = 0
  df_existing = pd.DataFrame(columns=["epoch", "loss", "accuracy"])

else:
  df_existing = pd.read_csv(csv_filename)
  if not df_existing.empty:
    last_epoch = int(df_existing["epoch"].max())
    start_epoch = last_epoch + 1
  else:
    start_epoch = 0
    df_existing = pd.DataFrame(columns=["epoch", "loss", "accuracy"])

In [3]:
train_transofrms = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomGrayscale(0.1),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [4]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    transform=train_transofrms,
    download=True
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    transform=test_transforms,
    download=True
)

In [5]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size = 256,
    shuffle = True,
    num_workers = 2,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size = 256,
    shuffle = False,
    num_workers = 2,
)

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"learning on: {device}")

if fine_tune == False:
    model = CifarNeural().to(device)
    
elif fine_tune == True:
    model = CifarNeural()
    model.load_state_dict(torch.load("modelCIFAR.pth"))
    model = model.to(device)

criterion = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.0001)

learning on: mps


In [7]:
EPOCHS = 100
best_accuracy = 0.0
history = []

for epoch in range(start_epoch, start_epoch + EPOCHS):
    model.train()

    for batch, (X, y) in enumerate(train_loader):
        X = X.to(device)
        y = y.to(device)

        preds = model(X)
        loss = criterion(preds, y)

        optim.zero_grad()

        loss.backward()

        optim.step()

    test_loss = 0
    correct_predictions = 0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for X, y in test_loader:
            X = X.to(device)
            y = y.to(device)

            preds = model(X)
            loss = criterion(preds, y)

            test_loss += loss.item()

            predicted_classes = preds.argmax(dim=1)
            correct_predictions += (predicted_classes == y).sum().item()
            total_samples += y.size(0)

        avg_test_loss = test_loss / len(test_loader)
        test_accuracy = correct_predictions / total_samples

        print(f"Epoch: {epoch}  | Loss: {avg_test_loss:.4f} | Accuracy: {test_accuracy * 100:.2f}%")

        new_row = pd.DataFrame(
            [{
                "epoch": epoch + 1,
                "loss": float(avg_test_loss),
                "accuracy": test_accuracy,
            }]
        )

        df_existing = pd.concat([df_existing, new_row], ignore_index=True)
        df_existing.to_csv(csv_filename, index=False)

        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            torch.save(model.state_dict(), "modelCIFAR_best.pth")
            print(f"--> Saved new best model with accuracy: {best_accuracy * 100:.2f}%")

Epoch: 0  | Loss: 1.2827 | Accuracy: 52.82%
--> Saved new best model with accuracy: 52.82%
Epoch: 1  | Loss: 1.0852 | Accuracy: 61.44%
--> Saved new best model with accuracy: 61.44%
Epoch: 2  | Loss: 0.9200 | Accuracy: 67.29%
--> Saved new best model with accuracy: 67.29%
Epoch: 3  | Loss: 0.7541 | Accuracy: 73.30%
--> Saved new best model with accuracy: 73.30%
Epoch: 4  | Loss: 0.6721 | Accuracy: 76.75%
--> Saved new best model with accuracy: 76.75%
Epoch: 5  | Loss: 0.6910 | Accuracy: 76.43%
Epoch: 6  | Loss: 0.5490 | Accuracy: 81.19%
--> Saved new best model with accuracy: 81.19%
Epoch: 7  | Loss: 0.5723 | Accuracy: 80.43%
Epoch: 8  | Loss: 0.6239 | Accuracy: 79.21%
Epoch: 9  | Loss: 0.5406 | Accuracy: 81.95%
--> Saved new best model with accuracy: 81.95%
Epoch: 10  | Loss: 0.5149 | Accuracy: 82.07%
--> Saved new best model with accuracy: 82.07%
Epoch: 11  | Loss: 0.4614 | Accuracy: 84.84%
--> Saved new best model with accuracy: 84.84%
Epoch: 12  | Loss: 0.4287 | Accuracy: 85.35%
--

KeyboardInterrupt: 